[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/zhimingkuang/Harvard-AM-115/blob/main/10_markov_chain/estimateMarkovChain.ipynb
)

# Estimate a Markov Chain from Data

In [ ]:
import numpy as np

Load data
> If working within Google Colab, run the following cell to clone the Github repo to access the data. If not can just skip to the next cell.






In [ ]:
# If using Google Colab, run these lines to import data from github
#!git clone https://github.com/zhimingkuang/Harvard-AM-115.git
#%cd Harvard-AM-115/10_markov_chain

In [ ]:
# ---- Read sequence (keep only H/C in case the file has newlines/spaces) ----
with open("weather_markov_sequence.txt", "r") as f:
    text = f.read()
seq = "".join(ch for ch in text if ch in ("H", "C"))
chars = np.array(list(seq))

# Map states to numbers: H = 0, C = 1
num_seq = np.full(len(chars), -9999, dtype=int)
num_seq[chars == "H"] = 0
num_seq[chars == "C"] = 1

Transition count matrix

In [ ]:
# ---- Transition count matrix T (rows: To state, cols: From state) ----
T = np.zeros((2, 2), dtype=int)

for t in range(len(num_seq) - 1):
    j = num_seq[t]      # from-state index (0=H, 1=C)
    i = num_seq[t + 1]  # to-state index
    T[i, j] += 1

print("Transition Count Matrix:")
print("      From H     From C")
print(f"To H   {T[0,0]}       {T[0,1]}")
print(f"To C   {T[1,0]}       {T[1,1]}")

Bootstrap

In [ ]:
# ---- Bootstrap ----
B = 5000  # number of bootstrap trials

# Build vectors of next-states for each from-state
nextFromH = num_seq[1:][num_seq[:-1] == 0]
nextFromC = num_seq[1:][num_seq[:-1] == 1]

NH = len(nextFromH)  # number of cases starting from Hot
NC = len(nextFromC)  # number of cases starting from Cold

# (optional) for reproducibility
# np.random.seed(42)

pHH_boot = np.zeros(B, dtype=float)
pCH_boot = np.zeros(B, dtype=float)

for b in range(B):
    sampH = nextFromH[np.random.randint(0, NH, size=NH)]
    sampC = nextFromC[np.random.randint(0, NC, size=NC)]

    pHH_boot[b] = np.mean(sampH == 0)  # P(H->H)
    pCH_boot[b] = np.mean(sampC == 0)  # P(C->H)

ciHH = np.quantile(pHH_boot, [0.025, 0.975])
ciCH = np.quantile(pCH_boot, [0.025, 0.975])

print(f"Bootstrap 95% CI p(H->H): [{ciHH[0]:.4f}, {ciHH[1]:.4f}]")
print(f"Bootstrap 95% CI p(C->H): [{ciCH[0]:.4f}, {ciCH[1]:.4f}]")